# Notebook 009: Deduplication Strategies: dropDuplicates vs Window Row-Number

**Module:** Module 01: Core PySpark Foundations & DataFrame API  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 11: Auto-loader & Video 23: Interview Questions  

---

## 1. Business Scenario & Real-World Context
An e-commerce order ingestion pipeline receives duplicate HTTP webhook payloads. Some duplicates are exact identical records, while others represent updated state payloads for the same order with later timestamps.

## 2. Technical Requirement & Problem Statement
Implement deduplication: (1) `distinct()` for exact row matches, (2) `dropDuplicates([subset])` for primary key deduplication with arbitrary winner selection, and (3) Deterministic deduplication using `row_number()` over Window partitions to retain the latest timestamp record.

## 3. Spark & Databricks Architecture Notes
`dropDuplicates()` uses a HashAggregate under the hood. While efficient, it is non-deterministic regarding which non-key column values survive if not pre-sorted. Window functions (`row_number`) provide 100% deterministic deduplication by explicitly controlling sorting criteria.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_009_Deduplication_Strategies_dropDuplicates_vs_Window_Row-Number")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Generating Webhook Ingestion Duplicates
Create dataset containing both exact duplicates and out-of-order state updates.

In [ ]:
webhook_events = [
    ("ORD-101", "CREATED", "2023-11-20 09:00:00", 120.00),
    ("ORD-101", "PAID",    "2023-11-20 09:05:00", 120.00),
    ("ORD-101", "PAID",    "2023-11-20 09:05:00", 120.00), # Exact duplicate
    ("ORD-102", "CREATED", "2023-11-20 09:10:00", 85.50),
    ("ORD-102", "CANCELLED","2023-11-20 09:12:00", 85.50),
    ("ORD-103", "CREATED", "2023-11-20 09:15:00", 430.00)
]

schema = "order_id STRING, order_status STRING, updated_at STRING, total_amount DOUBLE"
df_webhooks = spark.createDataFrame(webhook_events, schema)
df_webhooks.show()

## 6. Strategy 1 & 2: distinct() and dropDuplicates()
Demonstrate the limitations of simple deduplication.

In [ ]:
# Distinct drops identical rows
df_distinct = df_webhooks.distinct()
print(f"Original: {df_webhooks.count()} | After distinct: {df_distinct.count()}")

# dropDuplicates on order_id keeps an arbitrary update
df_arbitrary = df_webhooks.dropDuplicates(["order_id"])
print("Arbitrary dropDuplicates (non-deterministic latest status):")
df_arbitrary.show()

## 7. Strategy 3: Deterministic Window Deduplication
Partition by primary key, order by timestamp descending, and filter where row_number == 1.

In [ ]:
window_spec = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())

df_deterministic = (
    df_webhooks
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

print("Deterministic Deduplication (Always Keeps Latest Status):")
df_deterministic.orderBy("order_id").show()

## 9. Key Architectural Takeaways
- **`distinct()` hashes all columns across the dataset, triggering a shuffle.**
- **`dropDuplicates(subset)` is faster than window functions but does not allow ordering criteria for which duplicate record survives.**
- **For event-driven CDC feeds, always use `row_number().over(Window.partitionBy(pk).orderBy(F.col('ts').desc())) == 1`.**

## 10. Senior Data Engineering Interview Questions
1. Why can `dropDuplicates(['id'])` return non-deterministic values for remaining columns?
2. What is the memory and shuffle difference between `dropDuplicates` and `row_number()` window filtering?
3. How does Delta Lake `MERGE INTO` replace the need for batch deduplication in medallion Silver tables?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
